# 00 - Session bootstrap (smoke test)

**What this notebook does:** runs `scripts/bootstrap_session.py`, which detects the
platform (Colab / Kaggle / local), mounts Drive on Colab, reads the GitHub PAT from
the host secret store, clones or hard-resets this repo to the requested branch,
installs `requirements-colab.txt` only if imports are missing, and symlinks
`data/`, `outputs/` and `checkpoints/` to their persistent locations. It then
prints a session summary and nothing else.

**What must already exist:**
- a GitHub PAT stored as a secret named `GH_TOKEN` (Colab: Secrets panel; Kaggle: Add-ons -> Secrets)
- `configs/default.yaml` filled in for `session.*` (repo url, branch, Colab Drive paths, Kaggle dataset slug)
- on Kaggle: the dataset attached to the notebook; on Colab: the dataset present in Drive

**What it produces:** a cloned/updated repo, resolved persistent directories, and a
printed summary (platform, GPU, VRAM, torch version, repo commit, data root, persistent dir).
No model, no data written.

**Expected runtime on a free T4:** ~1-2 min on first run (clone + pip), a few
seconds on re-runs.

## Cell 1 - strip stored outputs

Notebooks in this repo are committed **without** stored outputs. Run this cell
right before committing (it is safe to run at any time and does nothing else).

In [ ]:
import glob, subprocess, sys

_nb = sorted(glob.glob('**/00_bootstrap.ipynb', recursive=True))
if _nb:
    subprocess.run(
        [sys.executable, '-m', 'jupyter', 'nbconvert', '--clear-output', '--inplace', _nb[0]],
        check=False,
    )
    print('cleared outputs in', _nb[0])

## Cell 2 - fetch the bootstrap script

On a fresh session the repo is not cloned yet, so this cell downloads only
`bootstrap_session.py` from the public raw URL. If we are already running inside
a checkout the local copy is used instead. This cell contains no token and no
personal path - the PAT is read by the script from the host secret store.

In [ ]:
import os, sys, urllib.request

REPO_URL = 'https://github.com/arhorri/boundary.git'
BRANCH = 'main'
RAW_URL = f'https://raw.githubusercontent.com/arhorri/boundary/{BRANCH}/scripts/bootstrap_session.py'

if os.path.isfile('scripts/bootstrap_session.py'):
    sys.path.insert(0, 'scripts')
else:
    urllib.request.urlretrieve(RAW_URL, 'bootstrap_session.py')
    sys.path.insert(0, os.getcwd())

import bootstrap_session

## Cell 3 - run the bootstrap

This performs every idempotent step and prints the one-line GPU/VRAM summary
plus the full session summary. Re-run it after any disconnect. `paths` holds
the resolved locations that the rest of the pipeline imports via `src.paths`.

In [ ]:
paths = bootstrap_session.bootstrap(repo_url=REPO_URL, branch=BRANCH)
paths